# 🧠 2 · Train your voice model

Trains an **RVC** voice model from the dataset made in notebook 1, using [Applio](https://github.com/IAHispano/Applio).
Output: `MyVoice/models/<name>/<name>.pth` + `.index` on your Drive.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).
Expect roughly 30-90 minutes for 300 epochs on a T4. Progress is backed up to Drive every 5 minutes, so if Colab disconnects,
just run all cells again with `RESUME_FROM_DRIVE` ticked.

In [ ]:
#@title 📁 Connect Google Drive and create folders
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

VOICE_ROOT = Path("/content/drive/MyDrive/MyVoice")   # everything lives here, so it survives Colab resets
FOLDERS = {name: VOICE_ROOT / name for name in
           ["recordings", "dataset", "models", "checkpoints", "tracks", "instrumentals", "vocals", "output"]}
for p in FOLDERS.values():
    p.mkdir(parents=True, exist_ok=True)
print("Your folders:")
for name, p in FOLDERS.items():
    print(f"  {name:14s} {p}")

In [ ]:
import os, re, shutil, subprocess, sys, time
from pathlib import Path

def run(cmd, cwd=None, env=None):
    """Run a command, stream its output live, and raise if it fails."""
    cmd = [str(c) for c in cmd]
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Command failed (exit {code}): {' '.join(cmd)}")

if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then re-run from the top.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 🔧 Install the voice engine (Applio / RVC) - about 5 min the first time
APPLIO_REPO = "https://github.com/IAHispano/Applio"
APPLIO_REF = "c7665ac9a305b3683570ed914f1577d53d4b75c4"   #@param {type:"string"}
# ^ a commit these notebooks were written against. Type "main" to try the newest Applio instead.
APPLIO_DIR = Path("/content/Applio")

if not APPLIO_DIR.exists():
    run(["git", "clone", "-q", APPLIO_REPO, APPLIO_DIR])
run(["git", "checkout", "-q", APPLIO_REF], cwd=APPLIO_DIR)

marker = APPLIO_DIR / ".installed"
if marker.exists() and marker.read_text() == APPLIO_REF:
    print("Applio already installed in this session.")
else:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "portaudio19-dev"])
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    run([sys.executable, "-m", "uv", "pip", "install", "--system", "-q", "-r", "requirements.txt",
         "--extra-index-url", "https://download.pytorch.org/whl/cu128",
         "--index-strategy", "unsafe-best-match"], cwd=APPLIO_DIR)
    run([sys.executable, "core.py", "prerequisites", "--models", "--pretraineds-hifigan", "--exe"], cwd=APPLIO_DIR)
    shutil.copy(APPLIO_DIR / "assets" / "config_template.json", APPLIO_DIR / "assets" / "config.json")
    marker.write_text(APPLIO_REF)
    print("Applio installed.")

In [ ]:
#@title ⚙️ Training settings
MODEL_NAME = "myvoice"     #@param {type:"string"}
SAMPLE_RATE = "40k"        #@param ["32k", "40k", "48k"]
TOTAL_EPOCHS = 300         #@param {type:"integer"}
SAVE_EVERY = 25            #@param {type:"slider", min:5, max:100, step:5}
BATCH_SIZE = 8             #@param {type:"slider", min:2, max:16, step:1}
NOISE_REDUCTION = False    #@param {type:"boolean"}
RESUME_FROM_DRIVE = True   #@param {type:"boolean"}
# - 300 epochs is a good start for 10-30 min of audio. Weights are saved every SAVE_EVERY epochs, so you can pick the best one later.
# - Out of memory on the GPU? lower BATCH_SIZE.  Colab disconnected mid-training? Re-run all cells with RESUME_FROM_DRIVE on.
# - 40k is a good all-round choice; pick 48k if your recordings are very clean and you want the most top-end detail.

assert re.fullmatch(r"[A-Za-z0-9_-]+", MODEL_NAME), "Use only letters, numbers, - and _ in MODEL_NAME"
SR = int(SAMPLE_RATE.rstrip("k")) * 1000
DATASET_DIR = FOLDERS["dataset"]
BACKUP_DIR = FOLDERS["checkpoints"] / MODEL_NAME
LOCAL_DATASET = Path("/content/dataset_local")
LOG_DIR = APPLIO_DIR / "logs" / MODEL_NAME

n_wavs = len(list(DATASET_DIR.glob("*.wav")))
if n_wavs == 0:
    raise SystemExit(f"No dataset found in {DATASET_DIR}. Run notebook 01 first.")
print(f"{n_wavs} dataset file(s) found. Training '{MODEL_NAME}' at {SAMPLE_RATE} for {TOTAL_EPOCHS} epochs.")

## Train

In [ ]:
#@title 1️⃣ Preprocess (slice the dataset into short clips)
if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)
shutil.copytree(DATASET_DIR, LOCAL_DATASET)          # local copy = faster than reading from Drive

if LOG_DIR.exists() and not RESUME_FROM_DRIVE:
    shutil.rmtree(LOG_DIR)                           # fresh start

cmd = [sys.executable, "core.py", "preprocess",
       "--model-name", MODEL_NAME, "--dataset-path", LOCAL_DATASET,
       "--sample-rate", SR, "--cpu-cores", min(os.cpu_count() or 2, 64),
       "--cut-preprocess", "Automatic", "--chunk-len", "3.0", "--overlap-len", "0.3",
       "--normalization-mode", "none"]
if NOISE_REDUCTION:
    cmd += ["--noise-reduction", "--noise-reduction-strength", "0.7"]
run(cmd, cwd=APPLIO_DIR)

In [ ]:
#@title 2️⃣ Extract pitch + voice features
run([sys.executable, "core.py", "extract", "--model-name", MODEL_NAME,
     "--f0-method", "rmvpe", "--sample-rate", SR, "--cpu-cores", min(os.cpu_count() or 2, 64),
     "--gpu", "0", "--embedder-model", "contentvec", "--include-mutes", 2], cwd=APPLIO_DIR)

In [ ]:
#@title 💾 Restore an earlier run (if any) and start auto-backup to Drive
import glob, threading

BACKUP_GLOBS = ["G_*.pth", "D_*.pth", f"{MODEL_NAME}_*e_*s.pth", "*.index", "config.json"]
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def sync(src_dir, dst_dir):
    """Copy new/changed checkpoint files from src_dir to dst_dir (atomic per file)."""
    copied = 0
    for pattern in BACKUP_GLOBS:
        for f in glob.glob(str(Path(src_dir) / pattern)):
            f = Path(f)
            dst = Path(dst_dir) / f.name
            if dst.exists() and dst.stat().st_size == f.stat().st_size and dst.stat().st_mtime >= f.stat().st_mtime:
                continue
            tmp = dst.with_name(dst.name + ".part")
            shutil.copy2(f, tmp)
            os.replace(tmp, dst)
            copied += 1
    return copied

if RESUME_FROM_DRIVE and any(BACKUP_DIR.glob("G_*.pth")):
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    for pattern in ["G_*.pth", "D_*.pth", f"{MODEL_NAME}_*e_*s.pth"]:
        for f in BACKUP_DIR.glob(pattern):
            shutil.copy2(f, LOG_DIR / f.name)
    print(f"♻️  Restored checkpoints from {BACKUP_DIR} - training will continue where it stopped.")
else:
    print("Starting from scratch (no earlier checkpoint restored).")

stop_backup = threading.Event()
def _loop():
    while not stop_backup.wait(300):                 # every 5 minutes
        try:
            sync(LOG_DIR, BACKUP_DIR)
        except Exception as e:                       # never kill training because of a Drive hiccup
            print("backup warning:", e)
threading.Thread(target=_loop, daemon=True).start()
print("Auto-backup running (every 5 min) ->", BACKUP_DIR)

In [ ]:
#@title 3️⃣ Train 🚀 (the slow part - keep this tab open)
cmd = [sys.executable, "core.py", "train", "--model-name", MODEL_NAME,
       "--save-every-epoch", SAVE_EVERY, "--save-only-latest", "--save-every-weights",
       "--total-epoch", TOTAL_EPOCHS, "--sample-rate", SR, "--batch-size", BATCH_SIZE,
       "--gpu", "0", "--vocoder", "HiFi-GAN", "--pretrained"]
try:
    run(cmd, cwd=APPLIO_DIR)
finally:
    n = sync(LOG_DIR, BACKUP_DIR)                    # final backup, even if training was interrupted
    print(f"Backed up {n} file(s) to {BACKUP_DIR}")

## Export and test

In [ ]:
#@title 📦 Export the voice model to Drive (MyVoice/models)
EXPORT_EPOCH = 0   #@param {type:"integer"}
# 0 = use the newest weights. To use an earlier epoch (e.g. if the newest sounds over-trained / robotic), type its number, e.g. 200.

weights = {}
for f in LOG_DIR.glob(f"{MODEL_NAME}_*e_*s.pth"):
    m = re.fullmatch(rf"{re.escape(MODEL_NAME)}_(\d+)e_(\d+)s\.pth", f.name)
    if m:
        weights[int(m.group(1))] = f
if not weights:
    raise SystemExit("No trained weights found yet. Run the training cell first.")
print("Available epochs:", sorted(weights))

epoch = EXPORT_EPOCH or max(weights)
if epoch not in weights:
    raise SystemExit(f"Epoch {epoch} not found. Pick one of {sorted(weights)}.")

index_file = LOG_DIR / f"{MODEL_NAME}.index"
if not index_file.exists():                           # happens if training was stopped early
    run([sys.executable, "core.py", "index", "--model-name", MODEL_NAME, "--index-algorithm", "Auto"], cwd=APPLIO_DIR)

dest = FOLDERS["models"] / MODEL_NAME
dest.mkdir(parents=True, exist_ok=True)
shutil.copy2(weights[epoch], dest / f"{MODEL_NAME}.pth")
shutil.copy2(index_file, dest / f"{MODEL_NAME}.index")
(dest / "info.txt").write_text(f"epoch={epoch}\nsample_rate={SAMPLE_RATE}\nsource={weights[epoch].name}\n")
print(f"✅ Exported epoch {epoch} -> {dest}")
print(*(p.name for p in dest.iterdir()), sep="\n")

In [ ]:
#@title 🎧 Quick test: run one of your own clips through the model
# This only checks that the model loads and produces sound. The real test is notebook 03.
from IPython.display import Audio, display

dest = FOLDERS["models"] / MODEL_NAME
sample = sorted(LOCAL_DATASET.glob("*.wav"))[0]
test_out = Path("/content/test_convert.wav")
run([sys.executable, "core.py", "infer", "--input-path", sample, "--output-path", test_out,
     "--pth-path", dest / f"{MODEL_NAME}.pth", "--index-path", dest / f"{MODEL_NAME}.index",
     "--pitch", 0, "--index-rate", 0.5, "--protect", 0.33, "--f0-method", "rmvpe", "--export-format", "WAV"],
    cwd=APPLIO_DIR)
print("Original:");  display(Audio(str(sample)))
print("Converted:"); display(Audio(str(test_out)))

### Tips for a better voice
- **Robotic / metallic / lisping?** Usually over-trained → set `EXPORT_EPOCH` to an earlier checkpoint.
- **Sounds like a different person?** Under-trained or too little audio → record more, train longer.
- **Breathy or noisy output?** Check the recordings (room noise, clipping) in notebook 1.
- Re-training with a changed dataset? Use a new `MODEL_NAME` (or untick `RESUME_FROM_DRIVE`) so old checkpoints and the old index aren't reused.

Next: **03_add_my_voice_to_tracks.ipynb**